In [ ]:
import pandas as pd
import matplotlib as plt
import numpy as np

# Load data into dfs

In [ ]:
hockey_scouting_notes = pd.read_csv('../data/hockey_scouting_notes.csv')   # joinable by international_id
contracts_competition = pd.read_pickle('../data/contracts_competition.pkl') # joinable by international_id
performance = pd.read_csv('../data/performance.tsv', sep='\t') # joinable by international_id

identity_card_0 = pd.read_csv('../data/identity_card_0.tsv', sep='\t', header=0, names=['international_id',
                                                                                'medical_id',
                                                                                'first_name',
                                                                                'last_name',
                                                                                'gender',
                                                                                'age',
                                                                                'birth_city',
                                                                                'nationality'])
identity_card_1 = pd.read_csv('../data/identity_card_1.csv')

medical_information = pd.read_excel('../data/medical_information.xlsx',
                                    skiprows=2,
                                    header=0,
                                    names=['medical_id',
                                           'height',
                                           'weight',
                                           'age_in_years',
                                           'shoe_size',
                                           'body_fat_percentage',
                                           'fitness_level',
                                           'sprint_time',
                                           'medical_information',
                                           'return_date',
                                           'physician_signature']).drop(columns=['shoe_size',
                                                                                 'return_date',
                                                                                 'physician_signature']) # joinable by medical_id

moms_notes = pd.read_json('../data/moms_notes.json')



# Data cleaning and processing

## Identity card

In [ ]:
identity_card_1['gender'].unique()

In [ ]:
def transfer_roman_to_int(roman_numeral):
    if pd.isna(roman_numeral) or not isinstance(roman_numeral, str):
        return roman_numeral

    roman_map = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
    total = 0
    prev_value = 0
    for char in reversed(roman_numeral.upper().strip()):
        value = roman_map.get(char, 0)
        if value < prev_value:
            total -= value
        else:
            total += value
        prev_value = value
    return total

identity_card_0['international_id']=identity_card_0['international_id'].apply(transfer_roman_to_int)


In [ ]:
# Take out the ius from last names an "us" from "male" first names
identity_card_0['last_name'] = identity_card_0['last_name'].apply(lambda x: x[:-3])
identity_card_0['first_name'] = identity_card_0.apply(lambda x: x['first_name'][:-2] if x['gender'] == 'male' else x['first_name'],axis=1)

In [ ]:
identity_card = pd.concat([identity_card_0,identity_card_1])

# international_id

# gender -> normalize
gender_map = {'f':'female',
              'female': 'female',

              'm': 'male',
              'male': 'male',

              'other': 'other'}

identity_card['gender'] = identity_card['gender'].map(gender_map)

# age (why is there someone 312 years old?)

# birth_city and nationality -> normalize
identity_card['birth_city'] = identity_card['birth_city'].str.lower().str.strip()
identity_card['nationality'] = identity_card['nationality'].str.lower().str.strip()


identity_card


### ID card cleanup actions and remarks

* International Ids from id_card_0 transfered from roman to arab numerals
* From id_card_0 take out "ius" in last name
* From id_card_0 take out "us" in first names of males (One guys name was Hilarious before and now its only Hilario, which is a bit weird but i don't even know with this dataset) :D
* Normalize gender mapping
* Normalize cities and countries to lowercase

Remarks
* Some age outliers, what to do with that??? diverse and synthesized dataset, leave old people in
* Birth cities and nationalities often don't match
* There are 37 random rows with no data apart from international_id and medical_id, I'd drop those.

TODO
* Drop brith city because nationality and birth city doesnt match, not important
* drop 37 random rows
* drop people older than the oldest person on earth

## Scouting notes

In [ ]:
hockey_scouting_notes

### Scouting notes cleanup actions and remarks

Remarks
* Again some missing values, would decide later based on the classification what to do with them
* There are 37 missing rows again, matching the ids of the ones before

TODOS
* delete 37 random rows
* fill NaN in scout_notes with "no notes"
* fill NaN in dominant_hand with "none"
* drop people with 0 years + veteran
* check if years_pro is more than years_played --> drop

## Contracts competition

In [ ]:
contracts_competition

In [ ]:
# Cleaning contracts_competition

# contracts_signed, jersey_number, draft_year, number_of_previous_teams -> convert to int
contracts_competition['contracts_signed'] = contracts_competition['contracts_signed'].astype("Int64")
contracts_competition['jersey_number'] = contracts_competition['jersey_number'].astype("Int64")
contracts_competition['draft_year'] = contracts_competition['draft_year'].astype("Int64")
contracts_competition['number_of_previous_teams'] = contracts_competition['number_of_previous_teams'].astype("Int64")

# jersey_number
contracts_competition

### Contracts, competition actions and remarks
* Change columns to int where it makes sense

Remarks
* Again some missing values in contracts_signed and number_of_previous_teams, would decide later based on the classification what to do with them
* There are 37 missing rows again, matching the ids of the ones before
* Outliers in draft_year, probably corresponding with old players in the first dataset
* Some crazy high outliers in salaries too
* After contracts are resolved, also worth to check contracts against the number of prevoius teams

## Performance

In [ ]:
performance["goals"] = performance["goals"].astype("Int64")
performance["assists"] = performance["assists"].astype("Int64")
performance["num_of_shots"] = performance["num_of_shots"].astype("Int64")
performance["shot_attempts"] = performance["shot_attempts"].astype("Int64")
performance["high_danger_shots"] = performance["high_danger_shots"].astype("Int64")
performance["medium_danger_shots"] = performance["medium_danger_shots"].astype("Int64")
performance["low_danger_shots"] = performance["low_danger_shots"].astype("Int64")
performance["winning_goals"] = performance["winning_goals"].astype("Int64")
performance["power_play_goals"] = performance["power_play_goals"].astype("Int64")
performance["puck_touches"] = performance["puck_touches"].astype("Int64")
performance["puck_recoveries"] = performance["puck_recoveries"].astype("Int64")
# performance["penalties_taken"] = performance["penalties_taken"].astype("Int64")
performance["goals_against_total"] = performance["goals_against_total"].astype("Int64")
performance["passes_attempted"] = performance["passes_attempted"].astype("Int64")
performance["passes_completed"] = performance["passes_completed"].astype("Int64")
performance["games_missed_due_to_injury"] = performance["games_missed_due_to_injury"].astype("Int64")

In [ ]:
performance['shooting_percentage'] = performance['goals']/performance['num_of_shots']*100

In [ ]:
performance['save_percentage'] = performance['save_percentage'].apply(lambda x: 100 if x > 100 else x)
performance['penality_minutes'] = np.where(performance['penalties_taken'] > 0, performance['penality_minutes'], 0)

In [ ]:
performance['puck_possession_time'] = performance['puck_possession_time']*60
performance['penalty_kill_time'] = performance['penalty_kill_time']*60
performance['power_play_time'] = performance['power_play_time']*60
performance['time_on_ice'] = performance['time_on_ice']*60*60


In [ ]:
performance.drop('puck_touches', axis=1, inplace=True)
performance.drop('time_between_penalties', axis=1, inplace=True)
performance.drop('goals_against_total', axis=1, inplace=True)

In [ ]:
print(performance[performance['winning_goals'] > performance['goals']]['international_id'].count())
print(performance[performance['power_play_time'] > performance['time_on_ice']]['international_id'].count())
print(performance[performance['puck_possession_time'] > performance['time_on_ice']]['international_id'].count())
print(performance[performance['penalty_kill_time'] > performance['time_on_ice']]['international_id'].count())

In [ ]:
performance = performance[performance['time_on_ice'] < 1000000]

In [ ]:
performance

### Performance actions and remarks

Remarks
* How is shooting_percentage calculated? The number doesn't make sense at all. -> Calculated properly
* danger shots add up to shot_attempts not num_of_shots. -> good to know
* There are some people with save_percentages higher than 100. -> set to 100
* There are some people with higher power_play_time than time_on_ice. Could be time_on_ice is in hours, where power_play is in seconds but seems weird to me. -> assumption: time_on_ice in hours, power_play_time
* Puck touches again don't make any sense, as 453 players have more goals than puck touches and 5932 have more shot attempts than puck touches. -> drop puck touches
* Puck recoveries also don't make any sense as there are 2650 players with more recoveries than puck touches... -> drop puck_touches
* 3922 players have higher puck_possesion time than time_on_ice -> assume puck_posession is minutes and change it to seconds
* 841 player have a higher penalty kill time than time_on_ice... -> change penalty kill time to seconds from minutes
* 5762 players with 0 penalties taken and non-0 penalty time. -> set penalty minutes to 0 when player doesn't have any penalties
* 5941 players with 0 penalties and non-0 time_between_penalties. -> drop time_between penalties, because it doesn't make sense, and is probably not too relevant
* 1314 players with higher goals agains on average than total goals against. -> drop_goals_against total, as the average is more important.
* 6513 players with higher passes attempted than puck touches -> puck touches dropoped
* 2 players with time on ice bigger than 1milion seconds, outliers -> removed
* Number of shots generally around 450, some lower outliers, maybe goalies

TODO
* normalizing save_percentages (max 100)
* assume that time_on_ice is in hours, and power_play_time is minutes and normalize into seconds
* drop puck touches

DONE HERE:
* Calculate shooting percentage properly instead of the nonsense
* Set max save_percentage to 100 from those who had more than 100
* Change time_on_ice from hours to seconds and power play time from minutes, puck_posession from minutes too
* dropped puck_touches, because it collided with many other columns
* Changed penalty kill time to seconds from minutes assuming
* droppend time_between_penalties column
* dropped 37 nan values

In [ ]:
# Clean medical information

# normalize height and weight?

# convert age to int

# remove percent symbol in body_fat_percentage 

# which columns are unnecessary?

medical_information

,medical_id,height,weight,age_in_years,body_fat_percentage,fitness_level,sprint_time,medical_information
0,368bcbfc-cc90-43da-841f-aa57d4d789c7,1.79m,82.4772kg,24.0,20.87%,Average,3.9727,NaN
1,f7164dc2-6646-420e-b6b5-42fc07c11f6d,180.1135cm,93.8985,29.0,23.38,NaN,4.1549,NaN
2,09c9de13-5d3a-4b75-a7ec-ce10bb32efdd,163.1483cm,79.2438,25.0,23.41,good,4.0825,NaN
3,be8969bf-6de2-49fd-b0e9-c014bdb970d6,1.82m,80.8913kg,29.0,21.28%,Good,3.5001,NaN
4,476cd0f0-0154-4e70-be7c-661007e7fcc4,167.7038cm,109.2016,25.0,31.71,excellent,3.7619,NaN
...,...,...,...,...,...,...,...,...
9995,d8686224-e95b-4da5-86a9-0f43a4f63f26,174.6442cm,90.3047,21.0,24.89,excellent,4.0910,NaN
9996,6d9916ac-307f-4ceb-8f7d-a284b76831bc,160.4894cm,84.1026,20.0,28.3,elite,4.3003,NaN
9997,bf144ec6-af6e-4a4f-9026-7f02999205d7,158.7063cm,77.7094,28.0,25.62,average,3.8575,NaN
9998,9ba4ccea-771d-4cfc-9172-d2bf922178f3,173.6693cm,80.162,31.0,25.19,Elite,3.6665,NaN
